In [2]:
from sklearn.model_selection import StratifiedKFold, train_test_split
import numpy as np, pandas as pd

df = pd.read_csv('conversion_data_train.csv')
y = df['converted']
X = df.drop(columns=['converted'])

# Types propres
X['new_user'] = X['new_user'].astype('category')

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# IQR sur train uniquement
q1, q3 = X_tr['age'].quantile(0.25), X_tr['age'].quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr

train_mask = (X_tr['age'] >= low) & (X_tr['age'] <= high)
X_tr, y_tr = X_tr.loc[train_mask], y_tr.loc[train_mask]

# on n’écrase pas X_te/y_te (pas de fit sur test)


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_tr.select_dtypes(exclude=[np.number]).columns.tolist()

num_pipe = Pipeline([('scaler', StandardScaler(with_mean=False))])  # with_mean=False si sparse ensuite
cat_pipe = Pipeline([('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))])

preprocess = ColumnTransformer(
    [('num', num_pipe, num_cols),
     ('cat', cat_pipe, cat_cols)],
    remainder='drop'
)


In [4]:
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from xgboost import XGBClassifier
import numpy as np

clf = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)

pipe = Pipeline([('pre', preprocess), ('clf', clf)])

# Probas CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
probas = cross_val_predict(pipe, X_tr, y_tr, cv=cv, method='predict_proba', n_jobs=-1)[:,1]

# Seuil qui maximise F1 sur train (CV out-of-fold)
prec, rec, thr = precision_recall_curve(y_tr, probas)
f1s = 2*prec*rec/(prec+rec+1e-9)
best_thr = thr[np.nanargmax(f1s[:-1])]  # thr a len-1 vs prec/rec

best_thr


0.39521846

In [5]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
  'clf__n_estimators': [200, 400, 600],
  'clf__max_depth': [3, 5, 7],
  'clf__learning_rate': [0.03, 0.05, 0.1],
  'clf__subsample': [0.7, 0.85, 1.0],
  'clf__colsample_bytree': [0.7, 0.9, 1.0],
  'clf__min_child_weight': [1, 3, 5],
  'clf__gamma': [0, 0.5, 1.0],
}

search = RandomizedSearchCV(
    Pipeline([('pre', preprocess), ('clf', XGBClassifier(random_state=42, n_jobs=-1))]),
    param_distributions=param_dist,
    n_iter=30, scoring='f1', cv=cv, n_jobs=-1, verbose=1, random_state=42
)
search.fit(X_tr, y_tr)

best_pipe = search.best_estimator_


Fitting 5 folds for each of 30 candidates, totalling 150 fits


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

# seuil choisi sur proba OOF
proba_val = cross_val_predict(best_pipe, X_tr, y_tr, cv=cv, method='predict_proba', n_jobs=-1)[:,1]

# recalcul du meilleur seuil (option: garde “best_thr” précédent si stable)
prec, rec, thr = precision_recall_curve(y_tr, proba_val)
f1s = 2*prec*rec/(prec+rec+1e-9)
thr_star = thr[np.nanargmax(f1s[:-1])]

y_pred_val = (proba_val >= thr_star).astype(int)
print(confusion_matrix(y_tr, y_pred_val))
print(classification_report(y_tr, y_pred_val, digits=3))
print("F1 (CV, seuil optimisé):", f1_score(y_tr, y_pred_val))


[[217771   1454]
 [  1855   5480]]
              precision    recall  f1-score   support

           0      0.992     0.993     0.992    219225
           1      0.790     0.747     0.768      7335

    accuracy                          0.985    226560
   macro avg      0.891     0.870     0.880    226560
weighted avg      0.985     0.985     0.985    226560

F1 (CV, seuil optimisé): 0.7680986754502768


In [17]:
importances = best_pipe.named_steps['clf'].feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df)

                    Feature  Importance
1  num__total_pages_visited    0.667089
5           cat__new_user_1    0.149846
2      cat__country_Germany    0.060870
3           cat__country_UK    0.052464
4           cat__country_US    0.048218
0                  num__age    0.015082
6        cat__source_Direct    0.003897
7           cat__source_Seo    0.002534


In [18]:
# Fit final sur tout le train (X_tr + X_te optionnellement si tu veux “all train”,
# mais garde une note claire de ce que tu as fait)
best_pipe.fit(X_tr, y_tr)

# Fixer le seuil final = thr_star (issu de CV)
test_df = pd.read_csv('conversion_data_test.csv')
test_df['new_user'] = test_df['new_user'].astype('category')
proba_test = best_pipe.predict_proba(test_df)[:,1]
pred_test = (proba_test >= thr_star).astype(int)

sub = pd.DataFrame({'converted': pred_test})
run_id = 'Thibaut-XGB-F1thr'
sub.to_csv(f'conversion_data_test_predictions_{run_id}.csv', index=False)
